In [0]:
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

stations_df = spark.read.table(f"{catalog}.bronze.stations")

clean_stations = stations_df.withColumn('station_id', when(trim(col('station_id')) == '', lit(None).cast('string')).otherwise(trim(upper(col('station_id')))))\
    .withColumn('station_name', trim(col("station_name")))\
    .withColumn('city', trim(initcap(col("city"))))\
    .withColumn('site_type', trim(upper(col("site_type"))))\
    .withColumn('opened_on', try_to_date(col("opened_on"), "MM/dd/yyyy"))\
    .select("station_id", "station_name", "city", "site_type", "opened_on")

clean_stations.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stations")

In [0]:
tariffs_df = spark.read.table(f"{catalog}.bronze.tariffs")

clean_tariffs = tariffs_df\
    .withColumn('connector_type', trim(upper(col("connector_type"))))\
    .withColumn('time_band', trim(upper(col("time_band"))))\
    .withColumn('rate_per_kwh', col('rate_per_kwh').cast('decimal(10,2)'))\
    .withColumn('idle_fee_per_min', col('idle_fee_per_min').cast('decimal(10,2)'))\
    .withColumn('free_idle_minutes', col('free_idle_minutes').cast('int'))\
    .select("connector_type", "time_band", "rate_per_kwh", "idle_fee_per_min", "free_idle_minutes")

clean_tariffs.write.mode("overwrite").saveAsTable(f"{catalog}.silver.tariffs")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Create silver table with Change Data Feed enabled (requirement 4)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.chargers
(
    charger_id STRING,
    station_id STRING,
    connector_type STRING,
    max_kw DECIMAL(6,1),
    is_active BOOLEAN,
    commissioned_on DATE,
    last_changed_snapshot_date DATE
) USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = true)
""")

# Read and clean bronze chargers
bronze_chargers = spark.read.table(f"{catalog}.bronze.chargers")

clean_chargers = bronze_chargers \
    .withColumn("charger_id", trim(upper(col("charger_id")))) \
    .withColumn("station_id", trim(upper(col("station_id")))) \
    .withColumn("connector_type", trim(upper(col("connector.type")))) \
    .withColumn("max_kw", col("connector.max_kw").cast("decimal(6,1)")) \
    .withColumn("is_active", col("is_active").cast("boolean")) \
    .withColumn("commissioned_on", try_to_date(col("commissioned_on"), "yyyy-MM-dd")) \
    .withColumn("last_changed_snapshot_date", try_to_date(col("snapshot_date"), "yyyy-MM-dd")) \
    .select("charger_id", "station_id", "connector_type", "max_kw", "is_active",
            "commissioned_on", "last_changed_snapshot_date")

# Keep only the latest snapshot per charger (one row per charger in source)
clean_chargers = clean_chargers \
    .withColumn("_rn", row_number().over(
        Window.partitionBy("charger_id").orderBy(col("last_changed_snapshot_date").desc()))) \
    .filter(col("_rn") == 1) \
    .drop("_rn")

clean_chargers.createOrReplaceTempView("src_chargers")

# MERGE: update only when a value OTHER than last_changed_snapshot_date changed (requirement 2)
# Insert new chargers; do NOT delete chargers missing from later snapshots (requirement 3)
spark.sql(f"""
MERGE INTO {catalog}.silver.chargers t
USING src_chargers s
ON t.charger_id = s.charger_id
WHEN MATCHED AND (
    t.station_id IS DISTINCT FROM s.station_id
    OR t.connector_type IS DISTINCT FROM s.connector_type
    OR t.max_kw IS DISTINCT FROM s.max_kw
    OR t.is_active IS DISTINCT FROM s.is_active
    OR t.commissioned_on IS DISTINCT FROM s.commissioned_on
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Create silver table with Change Data Feed
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.members (
    member_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    plan STRING,
    joined_on DATE,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
) USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = true)
""")

# --- Standardize bronze members ---
bronze_members = spark.read.table(f"{catalog}.bronze.members")

clean_members = bronze_members \
    .withColumn('member_id', when(trim(upper(col('member_id'))) == '', lit(None).cast('string')).otherwise(trim(upper(col('member_id'))))) \
    .withColumn('full_name', trim(col('full_name'))) \
    .withColumn('email', trim(lower(col('email')))) \
    .withColumn('phone', substring(regexp_replace(col('phone'), '[^0-9]', ''), -10, 10)) \
    .withColumn('city', trim(initcap(col('city')))) \
    .withColumn('plan', trim(upper(col('plan')))) \
    .withColumn('joined_on', coalesce(
        try_to_date(col('joined_on'), lit('yyyy-MM-dd')),
        try_to_date(col('joined_on'), lit('dd/MM/yyyy'))
    )) \
    .withColumn('snapshot_date', try_to_date(
        regexp_extract(col('_source_file'), 'members_(\\d{4}-\\d{2}-\\d{2})', 1), lit('yyyy-MM-dd')
    )) \
    .withColumn('effective_date', coalesce(col('change_effective_date').cast('date'), col('snapshot_date')))

# Rule 1: Deduplicate identical rows (identical rows count once)
clean_members = clean_members.dropDuplicates([
    'member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on', 'effective_date'
])

# Rule 2 & 5: Process in effective_date order; skip rows with no change after standardisation
w_order = Window.partitionBy('member_id').orderBy('effective_date')

clean_members = clean_members \
    .withColumn('_prev_name', lag('full_name').over(w_order)) \
    .withColumn('_prev_email', lag('email').over(w_order)) \
    .withColumn('_prev_phone', lag('phone').over(w_order)) \
    .withColumn('_prev_plan', lag('plan').over(w_order)) \
    .withColumn('_prev_city', lag('city').over(w_order)) \
    .withColumn('_no_change',
        when(col('_prev_name').isNull(), lit(False))
        .otherwise(
            (col('full_name') == col('_prev_name')) &
            (col('email') == col('_prev_email')) &
            (col('phone') == col('_prev_phone')) &
            (col('plan') == col('_prev_plan')) &
            (col('city') == col('_prev_city'))
        )
    ) \
    .filter(~col('_no_change'))

# Rule 3: A change of plan or city starts a new version
w_ver = Window.partitionBy('member_id').orderBy('effective_date')

clean_members = clean_members \
    .withColumn('_prev_plan_v', lag('plan').over(w_ver)) \
    .withColumn('_prev_city_v', lag('city').over(w_ver)) \
    .withColumn('_new_ver',
        when(col('_prev_plan_v').isNull()
             | (col('plan') != col('_prev_plan_v'))
             | (col('city') != col('_prev_city_v')), lit(1))
        .otherwise(lit(0))
    ) \
    .withColumn('version_group', sum('_new_ver').over(
        w_ver.rowsBetween(Window.unboundedPreceding, Window.currentRow)))

# Rule 4: Within a version group, take the latest row (overwrite name/email/phone)
w_latest = Window.partitionBy('member_id', 'version_group').orderBy(col('effective_date').desc())

clean_members = clean_members \
    .withColumn('_rn', row_number().over(w_latest)) \
    .filter(col('_rn') == 1)

# Set SCD Type 2 columns
w_vg = Window.partitionBy('member_id', 'version_group')
w_seq = Window.partitionBy('member_id').orderBy('version_group')

clean_members = clean_members \
    .withColumn('effective_from', min('effective_date').over(w_vg)) \
    .withColumn('_next_from', lead('effective_from').over(w_seq)) \
    .withColumn('effective_to', col('_next_from')) \
    .withColumn('is_current', when(col('effective_to').isNull(), lit(True)).otherwise(lit(False))) \
    .select('member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on',
            'effective_from', 'effective_to', 'is_current')

# Rule 6: Idempotent MERGE (re-processing the same file changes nothing)
clean_members.createOrReplaceTempView("src_members")

spark.sql(f"""
MERGE INTO {catalog}.silver.members t
USING src_members s
ON t.member_id = s.member_id AND t.effective_from = s.effective_from
WHEN MATCHED AND (
    t.full_name IS DISTINCT FROM s.full_name
    OR t.email IS DISTINCT FROM s.email
    OR t.phone IS DISTINCT FROM s.phone
    OR t.city IS DISTINCT FROM s.city
    OR t.plan IS DISTINCT FROM s.plan
    OR t.joined_on IS DISTINCT FROM s.joined_on
    OR t.effective_to IS DISTINCT FROM s.effective_to
    OR t.is_current IS DISTINCT FROM s.is_current
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT * FROM {catalog}.silver.members ORDER BY member_id, effective_from LIMIT 20"))

In [0]:
%sql
select count(*) from vp_dp20040222.silver.members

In [0]:
%sql
select count(*) from vp_dp20040222.silver.chargers

In [0]:
%sql
select count(*) from vp_dp20040222.silver.stations

In [0]:
%sql
select count(*) from vp_dp20040222.silver.tariffs